# 06 — Fama-French + Momentum Factor Data

**Goal:** Construct a monthly time series of asset-pricing factors used to
risk-adjust portfolio returns. The five Fama-French factors plus momentum
(FF5 + MOM) are the right-hand side of the alpha regression in H2a.

**Output:** `data/factors.parquet`
- One row per month-end date
- 156 monthly observations (Jan 2012 – Dec 2024)
- Six factors plus the risk-free rate

**Factors:**
- `mkt_rf` — market excess return
- `smb` — size factor (Fama & French, 2015)
- `hml` — value factor (Fama & French, 2015)
- `rmw` — profitability factor (Fama & French, 2015)
- `cma` — investment factor (Fama & French, 2015)
- `mom` — momentum factor (Carhart, 1997)
- `rf` — one-month T-bill rate

**Data source:** Kenneth R. French Data Library, accessed via WRDS
(`ff.fivefactors_monthly` and `ff.factors_monthly`).

**Note on units:** WRDS returns factors in decimal form (e.g., 0.05 for 5%).
No additional scaling needed; matches the decimal format of CRSP returns
in notebook 04.

In [2]:
import pandas as pd
from pathlib import Path
import wrds

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PROCESSED = Path.home() / "thesis" / "data"

In [3]:
db = wrds.Connection()

Enter your WRDS username [<wrds-username>]: <wrds-username>
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [4]:
ff5_query = """
    SELECT date, mktrf AS mkt_rf, smb, hml, rmw, cma, rf
    FROM   ff.fivefactors_monthly
    WHERE  date BETWEEN '2012-01-01' AND '2024-12-31'
    ORDER  BY date
"""

ff5 = db.raw_sql(ff5_query)
ff5["date"] = pd.to_datetime(ff5["date"])
print(f"FF5 rows: {len(ff5)}")
print(f"Date range: {ff5['date'].min().date()} to {ff5['date'].max().date()}")
print()
print("First 5 rows:")
print(ff5.head())
print()
print("Distribution summary:")
print(ff5.describe().round(4))

FF5 rows: 156
Date range: 2012-01-01 to 2024-12-01

First 5 rows:
        date  mkt_rf     smb     hml     rmw     cma      rf
0 2012-01-01  0.0505  0.0202 -0.0093 -0.0206 -0.0148     0.0
1 2012-02-01   0.044 -0.0167  0.0042 -0.0036 -0.0008     0.0
2 2012-03-01  0.0311 -0.0048  0.0119 -0.0061   0.007     0.0
3 2012-04-01 -0.0084  -0.005 -0.0077   0.013  0.0066     0.0
4 2012-05-01 -0.0617 -0.0008 -0.0095  0.0203  0.0236  0.0001

Distribution summary:
                                date  mkt_rf     smb     hml     rmw     cma      rf
count                            156   156.0   156.0   156.0   156.0   156.0   156.0
mean   2018-06-16 14:27:41.538461440  0.0113 -0.0013 -0.0008  0.0025 -0.0002  0.0011
min              2012-01-01 00:00:00 -0.1335 -0.0818 -0.1383 -0.0474 -0.0708     0.0
25%              2015-03-24 06:00:00 -0.0112 -0.0196  -0.019 -0.0123  -0.015     0.0
50%              2018-06-16 00:00:00  0.0149 -0.0005 -0.0036  0.0029 -0.0009  0.0002
75%              2021-09-08 12:00:0

In [6]:
mom_query = """
    SELECT date, umd AS mom
    FROM   ff.factors_monthly
    WHERE  date BETWEEN '2012-01-01' AND '2024-12-31'
    ORDER  BY date
"""

mom = db.raw_sql(mom_query)
mom["date"] = pd.to_datetime(mom["date"])
print(f"Momentum rows: {len(mom)}")
print(f"Date range: {mom['date'].min().date()} to {mom['date'].max().date()}")
print()
print("Distribution:")
print(mom["mom"].describe().round(4))
print()
print("First 5 rows:")
print(mom.head())

Momentum rows: 156
Date range: 2012-01-01 to 2024-12-01

Distribution:
count     156.0
mean     0.0019
std      0.0372
min     -0.1621
25%     -0.0202
50%      0.0044
75%      0.0253
max      0.0997
Name: mom, dtype: Float64

First 5 rows:
        date     mom
0 2012-01-01 -0.0801
1 2012-02-01 -0.0026
2 2012-03-01  0.0131
3 2012-04-01  0.0372
4 2012-05-01  0.0644


In [7]:
# Merge FF5 + momentum on date
factors = ff5.merge(mom, on="date", how="inner")

# Reorder columns logically
factors = factors[["date", "mkt_rf", "smb", "hml", "rmw", "cma", "mom", "rf"]]

# Convert dates to month-end so they align with CRSP returns
factors["date"] = factors["date"] + pd.offsets.MonthEnd(0)

# Add year/month for grouping later
factors["year"] = factors["date"].dt.year
factors["month"] = factors["date"].dt.month

print(f"Final factor panel: {len(factors)} rows")
print(f"Date range: {factors['date'].min().date()} to {factors['date'].max().date()}")
print()
print("First 5 rows:")
print(factors.head())
print()
print("Last 5 rows:")
print(factors.tail())

Final factor panel: 156 rows
Date range: 2012-01-31 to 2024-12-31

First 5 rows:
        date  mkt_rf     smb     hml     rmw     cma     mom      rf  year  month
0 2012-01-31  0.0505  0.0202 -0.0093 -0.0206 -0.0148 -0.0801     0.0  2012      1
1 2012-02-29   0.044 -0.0167  0.0042 -0.0036 -0.0008 -0.0026     0.0  2012      2
2 2012-03-31  0.0311 -0.0048  0.0119 -0.0061   0.007  0.0131     0.0  2012      3
3 2012-04-30 -0.0084  -0.005 -0.0077   0.013  0.0066  0.0372     0.0  2012      4
4 2012-05-31 -0.0617 -0.0008 -0.0095  0.0203  0.0236  0.0644  0.0001  2012      5

Last 5 rows:
          date  mkt_rf     smb     hml     rmw     cma     mom      rf  year  month
151 2024-08-31  0.0161 -0.0357  -0.011  0.0075  0.0079  0.0486  0.0048  2024      8
152 2024-09-30  0.0173  -0.009 -0.0277  0.0019 -0.0025 -0.0056   0.004  2024      9
153 2024-10-31   -0.01 -0.0086  0.0089 -0.0149  0.0102  0.0294  0.0039  2024     10
154 2024-11-30  0.0649  0.0465  0.0029 -0.0227 -0.0193  0.0101   0.004  2024 

In [8]:
output_path = DATA_PROCESSED / "factors.parquet"
factors.to_parquet(output_path, index=False)

check = pd.read_parquet(output_path)
print(f"Saved {len(check)} rows to {output_path.name}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print()
print("Final summary:")
print(check.describe().round(4))

Saved 156 rows to factors.parquet
File size: 14.3 KB

Final summary:
                                date  mkt_rf     smb     hml     rmw     cma     mom      rf       year     month
count                            156   156.0   156.0   156.0   156.0   156.0   156.0   156.0   156.0000  156.0000
mean   2018-07-16 01:04:36.923076864  0.0113 -0.0013 -0.0008  0.0025 -0.0002  0.0019  0.0011  2018.0000    6.5000
min              2012-01-31 00:00:00 -0.1335 -0.0818 -0.1383 -0.0474 -0.0708 -0.1621     0.0  2012.0000    1.0000
25%              2015-04-22 12:00:00 -0.0112 -0.0196  -0.019 -0.0123  -0.015 -0.0202     0.0  2015.0000    3.7500
50%              2018-07-15 12:00:00  0.0149 -0.0005 -0.0036  0.0029 -0.0009  0.0044  0.0002  2018.0000    6.5000
75%              2021-10-07 18:00:00  0.0348  0.0145  0.0147   0.013  0.0118  0.0253  0.0018  2021.0000    9.2500
max              2024-12-31 00:00:00  0.1358  0.0833  0.1286  0.0719  0.0773  0.0997  0.0048  2024.0000   12.0000
std                